# AgingClockBench — Quickstart

Run a benchmark comparison of PhenoAge and KDM on the bundled NHANES 2015–2018 sample in under 5 minutes.

**Prerequisites:** `pip install agingclockbench`

In [ ]:
from agingclockbench import PhenoAge, KDM, BenchmarkSuite
from agingclockbench.datasets import load_nhanes_sample
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

## 1. Load bundled NHANES data

The bundled dataset is a preprocessed NHANES 2015–2018 sample (~5,000 rows) with mortality linkage from NCHS.

In [ ]:
df = load_nhanes_sample()
print(f"Loaded {len(df)} participants")
print(f"Age range: {df.age.min():.0f}–{df.age.max():.0f} years")
df.head()

## 2. Compute biological ages with each clock

In [ ]:
phenoage = PhenoAge()
kdm = KDM()

results = {
    "PhenoAge": phenoage.transform(df),
    "KDM":      kdm.transform(df),
}

for name, res in results.items():
    print(f"{name}: mean biological age = {res.biological_ages.mean():.1f} yr "
          f"(accel = {res.accel.mean():.1f} yr)")

## 3. Run benchmark suite (mortality-linked validation)

In [ ]:
suite = BenchmarkSuite(mortality_col="mortstat", followup_col="permth_exm")
report = suite.run(df, results)

report.to_dataframe()

## 4. Interpret results

| Column | What it means |
|--------|---------------|
| Pearson r | Correlation of biological age with chronological age (target: ≥ 0.82 for PhenoAge per Levine 2018) |
| Mort HR | Hazard ratio per SD of age acceleration (target: ~1.08 per year per Levine 2018) |
| CV | Coefficient of variation — lower = more stable clock |

### Expected PhenoAge benchmarks (Levine 2018, NHANES)
- Pearson r with chronological age: **~0.93**
- Mortality HR per year of acceleration: **1.08 (95% CI: 1.07–1.09)**

In [ ]:
# Single patient example
patient = pd.DataFrame([{
    "age": 52,
    "albumin_g_dl": 4.3,
    "creatinine_mg_dl": 0.9,
    "glucose_mg_dl": 87.0,
    "crp_mg_l": 0.3,
    "lymphocyte_pct": 28.0,
    "mcv_fl": 90.0,
    "rdw_pct": 13.0,
    "alp_u_l": 65.0,
    "wbc_k_ul": 6.0,
}])

res = PhenoAge().transform(patient)
print(f"Chronological age: {patient.age.iloc[0]:.0f}")
print(f"Biological age:    {res.biological_ages.iloc[0]:.1f}")
print(f"Age acceleration:  {res.accel.iloc[0]:.1f} years")